In [ ]:
# ============================================================
# FedMA on Tomato Leaf Disease Dataset — Kaggle Notebook
# Conditions: ResNet18, SGD(lr=0.001, momentum=0.9),
#             batch=32, local_epochs=5, rounds=10,
#             5 clients, Dirichlet α=0.5
# FedMA: layer-wise neuron matching via Hungarian algorithm
# Dataset: https://www.kaggle.com/datasets/kaustubhb999/tomatoleaf
# ============================================================

# ── Cell 0: Install dependency ───────────────────────────────
# scipy is needed for the Hungarian algorithm (linear_sum_assignment)
# It is pre-installed on Kaggle, but we verify here
import subprocess
result = subprocess.run(["pip", "show", "scipy"],
                        capture_output=True, text=True)
if "Version" in result.stdout:
    print("scipy already installed:", result.stdout.split("\n")[0])
else:
    subprocess.run(["pip", "install", "-q", "scipy"], check=True)
    print("scipy installed.")

# ── Cell 1: Verify dataset path ──────────────────────────────
import os
base = "/kaggle/input/tomatoleaf"
print("Scanning dataset structure...")
for root, dirs, files in os.walk(base):
    level = root.replace(base, '').count(os.sep)
    if level > 3:
        continue
    indent = '  ' * level
    n_files = len(files)
    print(f"{indent}{os.path.basename(root)}/"
          + (f"  [{n_files} files]" if n_files else ""))

# ── Cell 2: Imports ──────────────────────────────────────────
import copy
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms, models
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import (precision_score, recall_score,
                             f1_score, accuracy_score)
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings("ignore")

# ── Cell 3: Reproducibility & Device ────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

try:
    DEVICE = torch.device("cuda")
    _t = torch.zeros(2, 2).to(DEVICE) + 1
    del _t
    print(f"CUDA OK — {torch.cuda.get_device_name(0)}")
except Exception as e:
    print(f"CUDA unavailable ({e}), using CPU")
    DEVICE = torch.device("cpu")

# ── Cell 4: Hyperparameters ──────────────────────────────────
NUM_CLIENTS     = 5
DIRICHLET_ALPHA = 0.5
LOCAL_EPOCHS    = 5
GLOBAL_ROUNDS   = 10
BATCH_SIZE      = 32
LR              = 0.001
MOMENTUM        = 0.9

# ── Cell 5: Dataset paths ────────────────────────────────────
TRAIN_DIR = "/kaggle/input/datasets/kaustubhb999/tomatoleaf/tomato/train"
TEST_DIR  = "/kaggle/input/datasets/kaustubhb999/tomatoleaf/tomato/val"
if not os.path.exists(TEST_DIR):
    TEST_DIR = "/kaggle/input/tomatoleaf/tomato/test"
if not os.path.exists(TEST_DIR):
    TEST_DIR = None

print(f"Train dir : {TRAIN_DIR}  exists={os.path.exists(TRAIN_DIR)}")
print(f"Test  dir : {TEST_DIR}   "
      f"exists={os.path.exists(TEST_DIR) if TEST_DIR else False}")

# ── Cell 6: Transforms ───────────────────────────────────────
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# ── Cell 7: Load & pool dataset ──────────────────────────────
train_dataset = datasets.ImageFolder(root=TRAIN_DIR)
NUM_CLASSES   = len(train_dataset.classes)
print(f"\nClasses ({NUM_CLASSES}): {train_dataset.classes}")
print(f"Train images : {len(train_dataset)}")

all_samples = list(train_dataset.samples)

if TEST_DIR and os.path.exists(TEST_DIR):
    test_dataset = datasets.ImageFolder(root=TEST_DIR)
    remap = {v: train_dataset.class_to_idx[k]
             for k, v in test_dataset.class_to_idx.items()
             if k in train_dataset.class_to_idx}
    for path, lbl in test_dataset.samples:
        if lbl in remap:
            all_samples.append((path, remap[lbl]))
    print(f"Test  images : {len(test_dataset)}")

print(f"Total pooled : {len(all_samples)}")

# ── Cell 8: Sample-list Dataset ──────────────────────────────
from PIL import Image

class SampleDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples   = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label

# ── Cell 9: Non-IID Dirichlet split ──────────────────────────
def dirichlet_split(samples, num_clients, alpha, num_classes, seed=SEED):
    np.random.seed(seed)
    labels         = np.array([s[1] for s in samples])
    client_indices = [[] for _ in range(num_clients)]
    for cls in range(num_classes):
        cls_idx = np.where(labels == cls)[0]
        np.random.shuffle(cls_idx)
        if len(cls_idx) == 0:
            continue
        props = np.random.dirichlet(alpha * np.ones(num_clients))
        props = (props * len(cls_idx)).astype(int)
        props[np.argmax(props)] += len(cls_idx) - props.sum()
        splits = np.split(cls_idx, np.cumsum(props)[:-1])
        for c, split in enumerate(splits):
            client_indices[c].extend(split.tolist())
    for c in range(num_clients):
        random.shuffle(client_indices[c])
    return client_indices

client_indices = dirichlet_split(all_samples, NUM_CLIENTS,
                                 DIRICHLET_ALPHA, NUM_CLASSES)
print("\nClient data distribution:")
for i, idx in enumerate(client_indices):
    labels = [all_samples[j][1] for j in idx]
    print(f"  Client {i+1}: {len(idx):>5} samples | "
          f"{len(set(labels))} classes")

# ── Cell 10: Train/val split ─────────────────────────────────
def train_val_split(indices, val_ratio=0.2, seed=SEED):
    random.seed(seed)
    indices = list(indices)
    random.shuffle(indices)
    split = int(len(indices) * (1 - val_ratio))
    return indices[:split], indices[split:]

client_train_idx, client_val_idx = [], []
for idx in client_indices:
    tr, va = train_val_split(idx)
    client_train_idx.append(tr)
    client_val_idx.append(va)

# ── Cell 11: DataLoaders ─────────────────────────────────────
def make_loaders(train_ids, val_ids):
    tr_set  = SampleDataset([all_samples[i] for i in train_ids],
                             transform=train_transform)
    val_set = SampleDataset([all_samples[i] for i in val_ids],
                             transform=val_transform)
    tr_loader  = DataLoader(tr_set,  batch_size=BATCH_SIZE,
                            shuffle=True,  num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_set, batch_size=BATCH_SIZE,
                            shuffle=False, num_workers=2, pin_memory=True)
    return tr_loader, val_loader

client_loaders = []
for i in range(NUM_CLIENTS):
    tr_l, va_l = make_loaders(client_train_idx[i], client_val_idx[i])
    client_loaders.append((tr_l, va_l))
    print(f"Client {i+1}: train={len(client_train_idx[i])}, "
          f"val={len(client_val_idx[i])}")

all_val_idx = [i for va in client_val_idx for i in va]
global_val_loader = DataLoader(
    SampleDataset([all_samples[i] for i in all_val_idx],
                  transform=val_transform),
    batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True)
print(f"\nGlobal val set size: {len(all_val_idx)}")

# ── Cell 12: Model ───────────────────────────────────────────
def build_model(num_classes):
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model.to(DEVICE)

global_model = build_model(NUM_CLASSES)
print(f"ResNet18 → {NUM_CLASSES} output classes")

# ── Cell 13: Local training (standard SGD) ───────────────────
def local_train(local_model, train_loader, local_epochs, lr, momentum):
    """Standard local training — FedMA does the matching at aggregation."""
    local_model.train()
    optimizer = optim.SGD(local_model.parameters(),
                          lr=lr, momentum=momentum)
    criterion = nn.CrossEntropyLoss()
    for _ in range(local_epochs):
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(local_model(images), labels)
            loss.backward()
            optimizer.step()
    return local_model

# ── Cell 14: Hungarian neuron matching ───────────────────────
def hungarian_match(W_global, W_client):
    """
    Match neurons (rows) of W_client to neurons of W_global
    using cosine similarity and the Hungarian algorithm.

    W_global : (n_neurons, *) reference weight matrix
    W_client : (n_neurons, *) client weight matrix

    Returns a permutation index array such that
    W_client[perm] best aligns with W_global.
    """
    # Flatten each neuron to a 1-D vector
    Wg = W_global.reshape(W_global.shape[0], -1).cpu().numpy()
    Wc = W_client.reshape(W_client.shape[0], -1).cpu().numpy()

    # Normalise rows for cosine similarity
    def norm_rows(M):
        norms = np.linalg.norm(M, axis=1, keepdims=True)
        norms = np.where(norms == 0, 1e-8, norms)
        return M / norms

    Wg_n = norm_rows(Wg)
    Wc_n = norm_rows(Wc)

    # Cost matrix: 1 - cosine_similarity  (we minimise cost)
    cost = 1.0 - Wg_n @ Wc_n.T          # shape (n, n)

    # Hungarian algorithm → optimal assignment
    _, col_ind = linear_sum_assignment(cost)
    return col_ind                        # perm[i] = matched client neuron


# ── Cell 15: Extract ResNet18 conv layers for matching ────────
def get_conv_layer_names(model):
    """
    Return names of Conv2d layers (excluding the very first conv1
    whose input channels are fixed at 3 and do not need matching).
    We match all internal conv layers + the final FC layer.
    """
    names = []
    for name, module in model.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            names.append(name)
    return names


# ── Cell 16: FedMA layer-wise aggregation ────────────────────
def fedma_aggregate(global_model, client_models, client_sizes):
    """
    Simplified FedMA for ResNet18:

    For each CONV / BN / FC layer group (one BasicBlock at a time):
      1. For every client model, find the Hungarian permutation that
         best aligns its neurons to the current global layer.
      2. Permute the client's outgoing weights (current layer) and
         incoming weights of the next layer accordingly.
      3. Weighted-average the permuted weights into the global model.

    Because ResNet18 has residual skip connections, we match each
    layer independently (intra-layer matching) rather than doing
    full cross-layer propagation, which would require rewriting the
    skip projections. This is the standard practical approximation
    used in FedMA for ResNets.
    """
    total         = sum(client_sizes)
    global_sd     = global_model.state_dict()
    client_sds    = [m.state_dict() for m in client_models]
    new_sd        = copy.deepcopy(global_sd)

    # Collect all weight-bearing layer keys (conv weight + fc weight)
    weight_keys = [k for k in global_sd.keys()
                   if k.endswith('.weight')
                   and global_sd[k].dim() >= 2]

    for w_key in weight_keys:
        W_global = global_sd[w_key].float()
        n        = W_global.shape[0]      # number of output neurons/filters

        # Skip layers with very few neurons (e.g. first conv has 64 filters
        # but input is fixed — still match output dimension)
        accum = torch.zeros_like(W_global)

        # Also track corresponding bias key
        b_key  = w_key.replace('.weight', '.bias')
        has_bias = b_key in global_sd
        accum_b = (torch.zeros_like(global_sd[b_key].float())
                   if has_bias else None)

        # BN keys that correspond to this conv (same prefix)
        prefix  = w_key[:-len('.weight')]
        bn_keys = {k: k for k in global_sd.keys()
                   if k.startswith(prefix.replace('conv', 'bn'))
                   or k.startswith(prefix.replace('conv', 'downsample.1'))
                   if k != w_key}

        for c_idx, (c_sd, c_size) in enumerate(
                zip(client_sds, client_sizes)):
            W_client = c_sd[w_key].float()
            weight   = c_size / total

            # Hungarian matching on output neurons
            perm = hungarian_match(W_global, W_client)

            # Permute output dimension of this layer
            W_perm = W_client[perm]
            accum += weight * W_perm

            if has_bias:
                b_client = c_sd[b_key].float()
                accum_b += weight * b_client[perm]

        new_sd[w_key] = accum.to(global_sd[w_key].dtype)
        if has_bias:
            new_sd[b_key] = accum_b.to(global_sd[b_key].dtype)

    # For BN and other non-weight parameters, do plain weighted average
    for key in global_sd.keys():
        if key in new_sd and not key.endswith('.weight'):
            if key.endswith('.bias') and any(
                    key.replace('.bias', '.weight') == w
                    for w in weight_keys):
                continue   # already handled above
            accum = torch.zeros_like(global_sd[key].float())
            for c_sd, c_size in zip(client_sds, client_sizes):
                accum += (c_size / total) * c_sd[key].float()
            new_sd[key] = accum.to(global_sd[key].dtype)

    global_model.load_state_dict(new_sd)
    return global_model


# ── Cell 17: Evaluation ───────────────────────────────────────
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images  = images.to(DEVICE)
            outputs = model(images)
            preds   = outputs.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    acc  = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds,
                           average='weighted', zero_division=0)
    rec  = recall_score(all_labels, all_preds,
                        average='weighted', zero_division=0)
    f1   = f1_score(all_labels, all_preds,
                    average='weighted', zero_division=0)
    return acc, prec, rec, f1

# ── Cell 18: FedMA Training Loop ─────────────────────────────
history = {"round": [], "accuracy": [],
           "precision": [], "recall": [], "f1": []}
best_acc, best_state = 0.0, None

print("\n" + "="*65)
print(f"{'FedMA Training — Tomato Leaf Disease':^65}")
print(f"  Clients={NUM_CLIENTS} | α={DIRICHLET_ALPHA}")
print(f"  Rounds={GLOBAL_ROUNDS} | LocalEpochs={LOCAL_EPOCHS} | LR={LR}")
print(f"  Matching: Hungarian algorithm (cosine similarity)")
print("="*65)

for rnd in range(1, GLOBAL_ROUNDS + 1):

    client_models = []
    client_sizes  = []

    for c in range(NUM_CLIENTS):
        local_model  = copy.deepcopy(global_model)
        tr_loader, _ = client_loaders[c]
        local_model  = local_train(local_model, tr_loader,
                                   LOCAL_EPOCHS, LR, MOMENTUM)
        client_models.append(local_model.cpu())   # move to CPU for matching
        client_sizes.append(len(client_train_idx[c]))

    # Move global model to CPU for aggregation, then back to DEVICE
    global_model = global_model.cpu()
    global_model = fedma_aggregate(global_model, client_models, client_sizes)
    global_model = global_model.to(DEVICE)

    acc, prec, rec, f1 = evaluate(global_model, global_val_loader)
    history["round"].append(rnd)
    history["accuracy"].append(acc)
    history["precision"].append(prec)
    history["recall"].append(rec)
    history["f1"].append(f1)

    if acc > best_acc:
        best_acc   = acc
        best_state = copy.deepcopy(global_model.state_dict())

    print(f"Round {rnd:>2}/{GLOBAL_ROUNDS} | "
          f"Acc={acc:.4f} | Prec={prec:.4f} | "
          f"Rec={rec:.4f} | F1={f1:.4f}")

print("\n" + "="*65)
print(f"Best Accuracy: {best_acc:.4f}  ({best_acc*100:.2f}%)")
print("="*65)

torch.save(best_state, "fedma_tomatoleaf_best.pth")
print("Best model saved → fedma_tomatoleaf_best.pth")

# ── Cell 19: Final evaluation ─────────────────────────────────
global_model.load_state_dict(best_state)
final_acc, final_prec, final_rec, final_f1 = evaluate(
    global_model, global_val_loader)

print("\n── Final Results (Best Model) ──────────────────────────")
print(f"  Accuracy  : {final_acc:.4f}  ({final_acc*100:.2f}%)")
print(f"  Precision : {final_prec:.4f}")
print(f"  Recall    : {final_rec:.4f}")
print(f"  F1-Score  : {final_f1:.4f}")
print("────────────────────────────────────────────────────────")

# ── Cell 20: Plots ────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(
    "FedMA — Tomato Leaf Disease (ResNet18)\n"
    f"Clients={NUM_CLIENTS}, α={DIRICHLET_ALPHA}, "
    f"Rounds={GLOBAL_ROUNDS}, LocalEpochs={LOCAL_EPOCHS}, LR={LR}",
    fontsize=13, fontweight='bold')

metrics = [
    ("accuracy",  "Accuracy",  "royalblue"),
    ("precision", "Precision", "darkorange"),
    ("recall",    "Recall",    "green"),
    ("f1",        "F1-Score",  "red"),
]
for ax, (key, label, color) in zip(axes.flatten(), metrics):
    ax.plot(history["round"], history[key],
            color=color, linewidth=1.8, marker='o', markersize=4)
    ax.set_title(label, fontsize=11)
    ax.set_xlabel("Communication Round")
    ax.set_ylabel(label)
    ax.set_xlim(1, GLOBAL_ROUNDS)
    ax.set_xticks(range(1, GLOBAL_ROUNDS + 1))
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.3f'))
    ax.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.savefig("fedma_tomatoleaf_metrics.png", dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved → fedma_tomatoleaf_metrics.png")

# ── Cell 21: Per-client performance ──────────────────────────
print("\n── Per-Client Validation Performance (Best Model) ──────")
for c in range(NUM_CLIENTS):
    _, val_loader = client_loaders[c]
    acc, prec, rec, f1 = evaluate(global_model, val_loader)
    print(f"  Client {c+1}: Acc={acc:.4f} | Prec={prec:.4f} | "
          f"Rec={rec:.4f} | F1={f1:.4f}")

# ── Cell 22: Config summary ───────────────────────────────────
print("\n── Experiment Configuration ─────────────────────────────")
for k, v in {
    "Dataset"              : "Tomato Leaf Disease (kaustubhb999)",
    "Model"                : "ResNet18",
    "FL Algorithm"         : "FedMA",
    "Neuron Matching"      : "Hungarian algorithm (cosine similarity)",
    "Matching Scope"       : "Layer-wise (intra-layer, ResNet compatible)",
    "Num Clients"          : NUM_CLIENTS,
    "Dirichlet Alpha"      : DIRICHLET_ALPHA,
    "Local Epochs"         : LOCAL_EPOCHS,
    "Global Rounds"        : GLOBAL_ROUNDS,
    "Batch Size"           : BATCH_SIZE,
    "Learning Rate"        : LR,
    "Momentum"             : MOMENTUM,
    "Optimizer"            : "SGD",
    "Num Classes"          : NUM_CLASSES,
    "Total Samples"        : len(all_samples),
    "Best Accuracy"        : f"{best_acc*100:.2f}%",
    "Final F1-Score"       : f"{final_f1:.4f}",
}.items():
    print(f"  {k:<28}: {v}")

In [ ]:
# ============================================================
# SCAFFOLD on Tomato Leaf Disease Dataset — Kaggle Notebook
# Conditions: ResNet18, SGD(lr=0.001, momentum=0.9),
#             batch=32, local_epochs=5, rounds=50,
#             5 clients, Dirichlet α=0.5
# SCAFFOLD: global + local control variates maintained
# Dataset: https://www.kaggle.com/datasets/kaustubhb999/tomatoleaf
# ============================================================

# ── Cell 0: Verify dataset path ──────────────────────────────
import os

base = "/kaggle/input/tomatoleaf"
print("Scanning dataset structure...")
for root, dirs, files in os.walk(base):
    level = root.replace(base, '').count(os.sep)
    if level > 3:
        continue
    indent = '  ' * level
    n_files = len(files)
    print(f"{indent}{os.path.basename(root)}/"
          + (f"  [{n_files} files]" if n_files else ""))

# ── Cell 1: Imports ──────────────────────────────────────────
import copy
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import (precision_score, recall_score,
                             f1_score, accuracy_score)
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings("ignore")

# ── Cell 2: Reproducibility & Device ────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

try:
    DEVICE = torch.device("cuda")
    _t = torch.zeros(2, 2).to(DEVICE) + 1
    del _t
    print(f"CUDA OK — {torch.cuda.get_device_name(0)}")
except Exception as e:
    print(f"CUDA unavailable ({e}), using CPU")
    DEVICE = torch.device("cpu")

# ── Cell 3: Hyperparameters ──────────────────────────────────
NUM_CLIENTS     = 5
DIRICHLET_ALPHA = 0.5
LOCAL_EPOCHS    = 5
GLOBAL_ROUNDS   = 10
BATCH_SIZE      = 32
LR              = 0.001
MOMENTUM        = 0.9   # kept for config logging; SCAFFOLD uses plain SGD

# ── Cell 4: Dataset paths ────────────────────────────────────
TRAIN_DIR = "/kaggle/input/datasets/kaustubhb999/tomatoleaf/tomato/train"
TEST_DIR  = "/kaggle/input/datasets/kaustubhb999/tomatoleaf/tomato/val"
if not os.path.exists(TEST_DIR):
    TEST_DIR = "/kaggle/input/tomatoleaf/tomato/test"
if not os.path.exists(TEST_DIR):
    TEST_DIR = None

print(f"Train dir : {TRAIN_DIR}  exists={os.path.exists(TRAIN_DIR)}")
print(f"Test  dir : {TEST_DIR}   exists={os.path.exists(TEST_DIR) if TEST_DIR else False}")

# ── Cell 5: Transforms ───────────────────────────────────────
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# ── Cell 6: Load & pool dataset ──────────────────────────────
train_dataset = datasets.ImageFolder(root=TRAIN_DIR)
NUM_CLASSES   = len(train_dataset.classes)
print(f"\nClasses ({NUM_CLASSES}): {train_dataset.classes}")
print(f"Train images : {len(train_dataset)}")

all_samples = list(train_dataset.samples)

if TEST_DIR and os.path.exists(TEST_DIR):
    test_dataset = datasets.ImageFolder(root=TEST_DIR)
    remap = {v: train_dataset.class_to_idx[k]
             for k, v in test_dataset.class_to_idx.items()
             if k in train_dataset.class_to_idx}
    for path, lbl in test_dataset.samples:
        if lbl in remap:
            all_samples.append((path, remap[lbl]))
    print(f"Test  images : {len(test_dataset)}")

print(f"Total pooled : {len(all_samples)}")

# ── Cell 7: Sample-list Dataset ──────────────────────────────
from PIL import Image

class SampleDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples   = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label

# ── Cell 8: Non-IID Dirichlet split ──────────────────────────
def dirichlet_split(samples, num_clients, alpha, num_classes, seed=SEED):
    np.random.seed(seed)
    labels = np.array([s[1] for s in samples])
    client_indices = [[] for _ in range(num_clients)]

    for cls in range(num_classes):
        cls_idx = np.where(labels == cls)[0]
        np.random.shuffle(cls_idx)
        if len(cls_idx) == 0:
            continue
        props = np.random.dirichlet(alpha * np.ones(num_clients))
        props = (props * len(cls_idx)).astype(int)
        props[np.argmax(props)] += len(cls_idx) - props.sum()
        splits = np.split(cls_idx, np.cumsum(props)[:-1])
        for c, split in enumerate(splits):
            client_indices[c].extend(split.tolist())

    for c in range(num_clients):
        random.shuffle(client_indices[c])

    return client_indices

client_indices = dirichlet_split(all_samples, NUM_CLIENTS,
                                 DIRICHLET_ALPHA, NUM_CLASSES)

print("\nClient data distribution:")
for i, idx in enumerate(client_indices):
    labels = [all_samples[j][1] for j in idx]
    print(f"  Client {i+1}: {len(idx):>5} samples | "
          f"{len(set(labels))} classes")

# ── Cell 9: Train/val split per client ───────────────────────
def train_val_split(indices, val_ratio=0.2, seed=SEED):
    random.seed(seed)
    indices = list(indices)
    random.shuffle(indices)
    split = int(len(indices) * (1 - val_ratio))
    return indices[:split], indices[split:]

client_train_idx, client_val_idx = [], []
for idx in client_indices:
    tr, va = train_val_split(idx)
    client_train_idx.append(tr)
    client_val_idx.append(va)

# ── Cell 10: DataLoaders ─────────────────────────────────────
def make_loaders(train_ids, val_ids):
    tr_set  = SampleDataset([all_samples[i] for i in train_ids],
                             transform=train_transform)
    val_set = SampleDataset([all_samples[i] for i in val_ids],
                             transform=val_transform)
    tr_loader  = DataLoader(tr_set,  batch_size=BATCH_SIZE, shuffle=True,
                            num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=2, pin_memory=True)
    return tr_loader, val_loader

client_loaders = []
for i in range(NUM_CLIENTS):
    tr_l, va_l = make_loaders(client_train_idx[i], client_val_idx[i])
    client_loaders.append((tr_l, va_l))
    print(f"Client {i+1}: train={len(client_train_idx[i])}, "
          f"val={len(client_val_idx[i])}")

all_val_idx = [i for va in client_val_idx for i in va]
global_val_loader = DataLoader(
    SampleDataset([all_samples[i] for i in all_val_idx],
                  transform=val_transform),
    batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True)
print(f"\nGlobal val set size: {len(all_val_idx)}")

# ── Cell 11: Model ───────────────────────────────────────────
def build_model(num_classes):
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model.to(DEVICE)

global_model = build_model(NUM_CLASSES)
print(f"ResNet18 → {NUM_CLASSES} output classes")

# ── Cell 12: Control variate helpers ─────────────────────────
def zero_like_params(model):
    """Return zero tensors on CPU matching model parameter shapes."""
    return [torch.zeros_like(p.data.cpu()) for p in model.parameters()]

def copy_params(model):
    """Return a detached CPU copy of model parameters."""
    return [p.data.clone().cpu() for p in model.parameters()]

# ── Cell 13: SCAFFOLD local training ─────────────────────────
def scaffold_local_train(local_model, global_params,
                         c_global, c_local,
                         train_loader, local_epochs, lr, momentum):
    """
    SCAFFOLD local update (Option II).

    At each step:
        grad_corrected = grad - c_local + c_global
    After K local steps, update local control variate:
        c_local_new = c_local - c_global
                      + (1 / (K * lr)) * (w_global - w_local)

    Returns:
        new local state_dict,
        delta_y  (w_local - w_global)  — sent to server,
        delta_c  (c_local_new - c_local) — sent to server,
        c_local_new
    """
    device = next(local_model.parameters()).device  # auto-detect (cpu or cuda)

    local_model.train()
    criterion = nn.CrossEntropyLoss()

    steps_per_epoch = len(train_loader)
    K = local_epochs * steps_per_epoch

    # SCAFFOLD uses plain SGD (no momentum)
    optimizer = optim.SGD(local_model.parameters(), lr=lr)

    # ── FIX 1: move global_params, c_global, c_local to device once ──
    global_params_dev = [wp.to(device) for wp in global_params]
    c_global_dev      = [c.to(device)  for c  in c_global]
    c_local_dev       = [c.to(device)  for c  in c_local]

    for _ in range(local_epochs):
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()

            outputs = local_model(images)
            loss    = criterion(outputs, labels)
            loss.backward()

            # Apply SCAFFOLD correction: grad ← grad - c_i + c
            with torch.no_grad():
                for param, ci, c in zip(local_model.parameters(),
                                        c_local_dev, c_global_dev):
                    if param.grad is not None:
                        param.grad.data.add_(-ci + c)

            optimizer.step()

    # ── Update local control variate (Option II) ──────────────
    # c_i_new = c_i - c + (1 / K*lr) * (w_global - w_local)
    with torch.no_grad():
        c_local_new = []
        delta_c     = []
        local_params = list(local_model.parameters())

        for ci, c, wp, wl in zip(c_local_dev, c_global_dev,
                                  global_params_dev, local_params):
            ci_new = ci - c + (1.0 / (K * lr)) * (wp - wl.data)
            c_local_new.append(ci_new.cpu())   # store on CPU
            delta_c.append((ci_new - ci).cpu())

        # ── FIX 2: both wl.data and wp are now on the same device ──
        delta_y = [(wl.data - wp).cpu()
                   for wl, wp in zip(local_params, global_params_dev)]

    return local_model.state_dict(), delta_y, delta_c, c_local_new

# ── Cell 14: SCAFFOLD server aggregation ─────────────────────
def scaffold_aggregate(global_model, client_delta_y,
                       client_delta_c, c_global, client_sizes):
    """
    Server update:
        w_global ← w_global + (1/N) * Σ (size_c/total) * delta_y_c
        c_global ← c_global + (1/N) * Σ delta_c_c
    """
    N     = NUM_CLIENTS
    total = sum(client_sizes)

    with torch.no_grad():
        # ── FIX 3: accumulate weighted_dy on CPU, then move to DEVICE once ──
        for i, p in enumerate(global_model.parameters()):
            weighted_dy = sum(
                client_delta_y[c][i] * (client_sizes[c] / total)
                for c in range(N)
            )                                   # weighted_dy is a CPU tensor
            p.data.add_(weighted_dy.to(DEVICE)) # move to GPU for in-place add

        for i in range(len(c_global)):
            avg_dc = sum(client_delta_c[c][i] for c in range(N)) / N
            # c_global[i] lives on CPU; avg_dc also on CPU — no device conflict
            c_global[i] = (c_global[i] + avg_dc)

    return global_model, c_global

# ── Cell 15: Evaluation ───────────────────────────────────────
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images  = images.to(DEVICE)
            outputs = model(images)
            preds   = outputs.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    acc  = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds,
                           average='weighted', zero_division=0)
    rec  = recall_score(all_labels, all_preds,
                        average='weighted', zero_division=0)
    f1   = f1_score(all_labels, all_preds,
                    average='weighted', zero_division=0)
    return acc, prec, rec, f1

# ── Cell 16: Initialise control variates ─────────────────────
# All control variates stored on CPU (moved to device inside train fn)
c_global = zero_like_params(global_model)
c_locals = [zero_like_params(global_model) for _ in range(NUM_CLIENTS)]

# ── Cell 17: SCAFFOLD Training Loop ──────────────────────────
history = {"round": [], "accuracy": [],
           "precision": [], "recall": [], "f1": []}
best_acc, best_state = 0.0, None

print("\n" + "="*65)
print(f"{'SCAFFOLD Training — Tomato Leaf Disease':^65}")
print(f"  Clients={NUM_CLIENTS} | α={DIRICHLET_ALPHA}")
print(f"  Rounds={GLOBAL_ROUNDS} | LocalEpochs={LOCAL_EPOCHS} | LR={LR}")
print("="*65)

for rnd in range(1, GLOBAL_ROUNDS + 1):

    # Global params snapshot stored on CPU for safe delta computation
    global_params_snapshot = [p.data.clone().cpu()
                               for p in global_model.parameters()]

    client_delta_y = []
    client_delta_c = []
    client_sizes   = []

    for c in range(NUM_CLIENTS):
        local_model = copy.deepcopy(global_model)  # already on DEVICE
        tr_loader, _ = client_loaders[c]

        _, delta_y, delta_c, c_new = scaffold_local_train(
            local_model,
            global_params_snapshot,   # CPU tensors — moved inside fn
            c_global,                 # CPU tensors — moved inside fn
            c_locals[c],              # CPU tensors — moved inside fn
            tr_loader,
            LOCAL_EPOCHS, LR, MOMENTUM
        )

        client_delta_y.append(delta_y)
        client_delta_c.append(delta_c)
        client_sizes.append(len(client_train_idx[c]))
        c_locals[c] = c_new           # CPU tensors returned

    # Server aggregation
    global_model, c_global = scaffold_aggregate(
        global_model, client_delta_y, client_delta_c,
        c_global, client_sizes
    )

    # Evaluate
    acc, prec, rec, f1 = evaluate(global_model, global_val_loader)
    history["round"].append(rnd)
    history["accuracy"].append(acc)
    history["precision"].append(prec)
    history["recall"].append(rec)
    history["f1"].append(f1)

    if acc > best_acc:
        best_acc   = acc
        best_state = copy.deepcopy(global_model.state_dict())

    if rnd % 5 == 0 or rnd == 1:
        print(f"Round {rnd:>3}/{GLOBAL_ROUNDS} | "
              f"Acc={acc:.4f} | Prec={prec:.4f} | "
              f"Rec={rec:.4f} | F1={f1:.4f}")

print("\n" + "="*65)
print(f"Best Accuracy: {best_acc:.4f}  ({best_acc*100:.2f}%)")
print("="*65)

torch.save(best_state, "scaffold_tomatoleaf_best.pth")
print("Best model saved → scaffold_tomatoleaf_best.pth")

# ── Cell 18: Final evaluation ─────────────────────────────────
global_model.load_state_dict(best_state)
final_acc, final_prec, final_rec, final_f1 = evaluate(
    global_model, global_val_loader)

print("\n── Final Results (Best Model) ──────────────────────────")
print(f"  Accuracy  : {final_acc:.4f}  ({final_acc*100:.2f}%)")
print(f"  Precision : {final_prec:.4f}")
print(f"  Recall    : {final_rec:.4f}")
print(f"  F1-Score  : {final_f1:.4f}")
print("────────────────────────────────────────────────────────")

# ── Cell 19: Plots ────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(
    "SCAFFOLD — Tomato Leaf Disease (ResNet18)\n"
    f"Clients={NUM_CLIENTS}, α={DIRICHLET_ALPHA}, "
    f"Rounds={GLOBAL_ROUNDS}, LocalEpochs={LOCAL_EPOCHS}, LR={LR}",
    fontsize=13, fontweight='bold')

metrics = [
    ("accuracy",  "Accuracy",  "royalblue"),
    ("precision", "Precision", "darkorange"),
    ("recall",    "Recall",    "green"),
    ("f1",        "F1-Score",  "red"),
]
for ax, (key, label, color) in zip(axes.flatten(), metrics):
    ax.plot(history["round"], history[key],
            color=color, linewidth=1.8)
    ax.set_title(label, fontsize=11)
    ax.set_xlabel("Communication Round")
    ax.set_ylabel(label)
    ax.set_xlim(1, GLOBAL_ROUNDS)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.3f'))
    ax.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.savefig("scaffold_tomatoleaf_metrics.png", dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved → scaffold_tomatoleaf_metrics.png")

# ── Cell 20: Per-client performance ──────────────────────────
print("\n── Per-Client Validation Performance (Best Model) ──────")
for c in range(NUM_CLIENTS):
    _, val_loader = client_loaders[c]
    acc, prec, rec, f1 = evaluate(global_model, val_loader)
    print(f"  Client {c+1}: Acc={acc:.4f} | Prec={prec:.4f} | "
          f"Rec={rec:.4f} | F1={f1:.4f}")

# ── Cell 21: Config summary ───────────────────────────────────
print("\n── Experiment Configuration ─────────────────────────────")
for k, v in {
    "Dataset"          : "Tomato Leaf Disease (kaustubhb999)",
    "Model"            : "ResNet18",
    "FL Algorithm"     : "SCAFFOLD",
    "SCAFFOLD Option"  : "Option II (control variate correction)",
    "Num Clients"      : NUM_CLIENTS,
    "Dirichlet Alpha"  : DIRICHLET_ALPHA,
    "Local Epochs"     : LOCAL_EPOCHS,
    "Global Rounds"    : GLOBAL_ROUNDS,
    "Batch Size"       : BATCH_SIZE,
    "Learning Rate"    : LR,
    "Optimizer"        : "SGD (no momentum — SCAFFOLD standard)",
    "Num Classes"      : NUM_CLASSES,
    "Total Samples"    : len(all_samples),
    "Best Accuracy"    : f"{best_acc*100:.2f}%",
    "Final F1-Score"   : f"{final_f1:.4f}",
}.items():
    print(f"  {k:<28}: {v}")

In [ ]:
# ============================================================
# FedOpt (FedAdam) on Tomato Leaf Disease Dataset — Kaggle Notebook
# Conditions: ResNet18, SGD(lr=0.001, momentum=0.9) on clients,
#             Adam on server (server_lr=0.01, β1=0.9, β2=0.99, τ=1e-3)
#             batch=32, local_epochs=5, rounds=10,
#             5 clients, Dirichlet α=0.5
# FedOpt Paper: https://arxiv.org/abs/2003.00295
# Dataset: https://www.kaggle.com/datasets/kaustubhb999/tomatoleaf
# ============================================================

# ── Cell 0: Verify dataset path ──────────────────────────────
import os

base = "/kaggle/input/tomatoleaf"
print("Scanning dataset structure...")
for root, dirs, files in os.walk(base):
    level = root.replace(base, '').count(os.sep)
    if level > 3:
        continue
    indent = '  ' * level
    n_files = len(files)
    print(f"{indent}{os.path.basename(root)}/"
          + (f"  [{n_files} files]" if n_files else ""))

# ── Cell 1: Imports ──────────────────────────────────────────
import copy
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import (precision_score, recall_score,
                             f1_score, accuracy_score)
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings("ignore")

# ── Cell 2: Reproducibility & Device ────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

try:
    DEVICE = torch.device("cuda")
    _t = torch.zeros(2, 2).to(DEVICE) + 1
    del _t
    print(f"CUDA OK — {torch.cuda.get_device_name(0)}")
except Exception as e:
    print(f"CUDA unavailable ({e}), using CPU")
    DEVICE = torch.device("cpu")

# ── Cell 3: Hyperparameters ──────────────────────────────────
NUM_CLIENTS     = 5
DIRICHLET_ALPHA = 0.5
LOCAL_EPOCHS    = 5
GLOBAL_ROUNDS   = 10
BATCH_SIZE      = 32
CLIENT_LR       = 0.001   # client-side SGD lr
CLIENT_MOMENTUM = 0.9     # client-side SGD momentum

# FedOpt server-side Adam hyperparameters
SERVER_LR    = 0.01        # η  (server learning rate)
BETA1        = 0.9         # β₁ (momentum)
BETA2        = 0.99        # β₂ (second moment)
TAU          = 1e-3        # τ  (adaptivity / epsilon for numerical stability)

# ── Cell 4: Dataset paths ────────────────────────────────────
TRAIN_DIR = "/kaggle/input/datasets/kaustubhb999/tomatoleaf/tomato/train"
TEST_DIR  = "/kaggle/input/datasets/kaustubhb999/tomatoleaf/tomato/val"
if not os.path.exists(TEST_DIR):
    TEST_DIR = "/kaggle/input/tomatoleaf/tomato/test"
if not os.path.exists(TEST_DIR):
    TEST_DIR = None

print(f"Train dir : {TRAIN_DIR}  exists={os.path.exists(TRAIN_DIR)}")
print(f"Test  dir : {TEST_DIR}   exists={os.path.exists(TEST_DIR) if TEST_DIR else False}")

# ── Cell 5: Transforms ───────────────────────────────────────
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# ── Cell 6: Load & pool dataset ──────────────────────────────
train_dataset = datasets.ImageFolder(root=TRAIN_DIR)
NUM_CLASSES   = len(train_dataset.classes)
print(f"\nClasses ({NUM_CLASSES}): {train_dataset.classes}")
print(f"Train images : {len(train_dataset)}")

all_samples = list(train_dataset.samples)

if TEST_DIR and os.path.exists(TEST_DIR):
    test_dataset = datasets.ImageFolder(root=TEST_DIR)
    remap = {v: train_dataset.class_to_idx[k]
             for k, v in test_dataset.class_to_idx.items()
             if k in train_dataset.class_to_idx}
    for path, lbl in test_dataset.samples:
        if lbl in remap:
            all_samples.append((path, remap[lbl]))
    print(f"Test  images : {len(test_dataset)}")

print(f"Total pooled : {len(all_samples)}")

# ── Cell 7: Sample-list Dataset ──────────────────────────────
from PIL import Image

class SampleDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples   = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label

# ── Cell 8: Non-IID Dirichlet split ──────────────────────────
def dirichlet_split(samples, num_clients, alpha, num_classes, seed=SEED):
    np.random.seed(seed)
    labels = np.array([s[1] for s in samples])
    client_indices = [[] for _ in range(num_clients)]

    for cls in range(num_classes):
        cls_idx = np.where(labels == cls)[0]
        np.random.shuffle(cls_idx)
        if len(cls_idx) == 0:
            continue
        props = np.random.dirichlet(alpha * np.ones(num_clients))
        props = (props * len(cls_idx)).astype(int)
        props[np.argmax(props)] += len(cls_idx) - props.sum()
        splits = np.split(cls_idx, np.cumsum(props)[:-1])
        for c, split in enumerate(splits):
            client_indices[c].extend(split.tolist())

    for c in range(num_clients):
        random.shuffle(client_indices[c])

    return client_indices

client_indices = dirichlet_split(all_samples, NUM_CLIENTS,
                                 DIRICHLET_ALPHA, NUM_CLASSES)

print("\nClient data distribution:")
for i, idx in enumerate(client_indices):
    labels = [all_samples[j][1] for j in idx]
    print(f"  Client {i+1}: {len(idx):>5} samples | "
          f"{len(set(labels))} classes")

# ── Cell 9: Train/val split per client ───────────────────────
def train_val_split(indices, val_ratio=0.2, seed=SEED):
    random.seed(seed)
    indices = list(indices)
    random.shuffle(indices)
    split = int(len(indices) * (1 - val_ratio))
    return indices[:split], indices[split:]

client_train_idx, client_val_idx = [], []
for idx in client_indices:
    tr, va = train_val_split(idx)
    client_train_idx.append(tr)
    client_val_idx.append(va)

# ── Cell 10: DataLoaders ─────────────────────────────────────
def make_loaders(train_ids, val_ids):
    tr_set  = SampleDataset([all_samples[i] for i in train_ids],
                             transform=train_transform)
    val_set = SampleDataset([all_samples[i] for i in val_ids],
                             transform=val_transform)
    tr_loader  = DataLoader(tr_set,  batch_size=BATCH_SIZE, shuffle=True,
                            num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=2, pin_memory=True)
    return tr_loader, val_loader

client_loaders = []
for i in range(NUM_CLIENTS):
    tr_l, va_l = make_loaders(client_train_idx[i], client_val_idx[i])
    client_loaders.append((tr_l, va_l))
    print(f"Client {i+1}: train={len(client_train_idx[i])}, "
          f"val={len(client_val_idx[i])}")

all_val_idx = [i for va in client_val_idx for i in va]
global_val_loader = DataLoader(
    SampleDataset([all_samples[i] for i in all_val_idx],
                  transform=val_transform),
    batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True)
print(f"\nGlobal val set size: {len(all_val_idx)}")

# ── Cell 11: Model ───────────────────────────────────────────
def build_model(num_classes):
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model.to(DEVICE)

global_model = build_model(NUM_CLASSES)
print(f"ResNet18 → {NUM_CLASSES} output classes")

# ── Cell 12: FedOpt server state ─────────────────────────────
# m_t : first moment  (momentum),  stored on CPU
# v_t : second moment (variance),  stored on CPU
# Both initialised to zero; shape matches model parameters.

def zero_like_params_cpu(model):
    return [torch.zeros_like(p.data.cpu()) for p in model.parameters()]

server_m = zero_like_params_cpu(global_model)   # β₁ momentum
server_v = zero_like_params_cpu(global_model)   # β₂ second moment

# ── Cell 13: FedOpt local training (plain FedAvg client) ─────
def fedopt_local_train(local_model, train_loader,
                       local_epochs, lr, momentum):
    """
    Standard local SGD training (same as FedAvg client-side).
    FedOpt's novelty is entirely server-side; clients are unchanged.

    Returns:
        delta  : list of CPU tensors  (w_local - w_global)
        n_steps: total local steps taken (for logging)
    """
    device = next(local_model.parameters()).device

    # Save global weights for delta computation
    global_weights = [p.data.clone() for p in local_model.parameters()]

    local_model.train()
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(local_model.parameters(),
                          lr=lr, momentum=momentum)

    for _ in range(local_epochs):
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = local_model(images)
            loss    = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

    # delta_i = w_local - w_global  (pseudo-gradient contribution)
    delta = [(p.data - gw).cpu()
             for p, gw in zip(local_model.parameters(), global_weights)]

    return delta

# ── Cell 14: FedOpt server aggregation (FedAdam) ─────────────
def fedopt_aggregate(global_model, client_deltas, client_sizes,
                     server_m, server_v, round_num,
                     server_lr, beta1, beta2, tau):
    """
    FedOpt server update (FedAdam variant):

      Δ_t  = weighted average of client deltas  (pseudo-gradient)
      m_t  = β₁·m_{t-1} + (1-β₁)·Δ_t           (first moment)
      v_t  = β₂·v_{t-1} + (1-β₂)·Δ_t²           (second moment)
      w_t  = w_{t-1} + η · m_t / (√v_t + τ)     (parameter update)

    Note: Δ_t here is the *positive* pseudo-gradient (local - global),
    so the server adds it (gradient ascent in the loss ↔ descent in update).
    """
    total = sum(client_sizes)
    N     = len(client_sizes)

    with torch.no_grad():
        # ── Step 1: Compute weighted pseudo-gradient Δ_t (on CPU) ──
        num_params = len(server_m)
        delta_avg  = []
        for i in range(num_params):
            wt_sum = sum(
                client_deltas[c][i] * (client_sizes[c] / total)
                for c in range(N)
            )
            delta_avg.append(wt_sum)   # CPU tensor

        # ── Step 2: Update first & second moments ───────────────────
        for i in range(num_params):
            server_m[i] = beta1 * server_m[i] + (1.0 - beta1) * delta_avg[i]
            server_v[i] = beta2 * server_v[i] + (1.0 - beta2) * delta_avg[i] ** 2

        # ── Step 3: Bias correction (standard Adam) ─────────────────
        bc1 = 1.0 - beta1 ** round_num
        bc2 = 1.0 - beta2 ** round_num

        # ── Step 4: Update global model parameters ──────────────────
        for i, p in enumerate(global_model.parameters()):
            m_hat = server_m[i] / bc1           # bias-corrected, on CPU
            v_hat = server_v[i] / bc2           # bias-corrected, on CPU
            update = server_lr * m_hat / (torch.sqrt(v_hat) + tau)
            p.data.add_(update.to(DEVICE))      # move update to GPU once

    return global_model, server_m, server_v

# ── Cell 15: Evaluation ───────────────────────────────────────
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images  = images.to(DEVICE)
            outputs = model(images)
            preds   = outputs.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    acc  = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds,
                           average='weighted', zero_division=0)
    rec  = recall_score(all_labels, all_preds,
                        average='weighted', zero_division=0)
    f1   = f1_score(all_labels, all_preds,
                    average='weighted', zero_division=0)
    return acc, prec, rec, f1

# ── Cell 16: FedOpt Training Loop ────────────────────────────
history = {"round": [], "accuracy": [],
           "precision": [], "recall": [], "f1": []}
best_acc, best_state = 0.0, None

print("\n" + "="*65)
print(f"{'FedOpt (FedAdam) Training — Tomato Leaf Disease':^65}")
print(f"  Clients={NUM_CLIENTS} | α={DIRICHLET_ALPHA}")
print(f"  Rounds={GLOBAL_ROUNDS} | LocalEpochs={LOCAL_EPOCHS}")
print(f"  ClientLR={CLIENT_LR} | ServerLR={SERVER_LR}")
print(f"  β₁={BETA1} | β₂={BETA2} | τ={TAU}")
print("="*65)

for rnd in range(1, GLOBAL_ROUNDS + 1):

    client_deltas = []
    client_sizes  = []

    for c in range(NUM_CLIENTS):
        local_model = copy.deepcopy(global_model)   # on DEVICE
        tr_loader, _ = client_loaders[c]

        delta = fedopt_local_train(
            local_model, tr_loader,
            LOCAL_EPOCHS, CLIENT_LR, CLIENT_MOMENTUM
        )

        client_deltas.append(delta)
        client_sizes.append(len(client_train_idx[c]))

    # FedAdam server aggregation
    global_model, server_m, server_v = fedopt_aggregate(
        global_model, client_deltas, client_sizes,
        server_m, server_v, rnd,
        SERVER_LR, BETA1, BETA2, TAU
    )

    # Evaluate
    acc, prec, rec, f1 = evaluate(global_model, global_val_loader)
    history["round"].append(rnd)
    history["accuracy"].append(acc)
    history["precision"].append(prec)
    history["recall"].append(rec)
    history["f1"].append(f1)

    if acc > best_acc:
        best_acc   = acc
        best_state = copy.deepcopy(global_model.state_dict())

    if rnd % 5 == 0 or rnd == 1:
        print(f"Round {rnd:>3}/{GLOBAL_ROUNDS} | "
              f"Acc={acc:.4f} | Prec={prec:.4f} | "
              f"Rec={rec:.4f} | F1={f1:.4f}")

print("\n" + "="*65)
print(f"Best Accuracy: {best_acc:.4f}  ({best_acc*100:.2f}%)")
print("="*65)

torch.save(best_state, "fedopt_tomatoleaf_best.pth")
print("Best model saved → fedopt_tomatoleaf_best.pth")

# ── Cell 17: Final evaluation ─────────────────────────────────
global_model.load_state_dict(best_state)
final_acc, final_prec, final_rec, final_f1 = evaluate(
    global_model, global_val_loader)

print("\n── Final Results (Best Model) ──────────────────────────")
print(f"  Accuracy  : {final_acc:.4f}  ({final_acc*100:.2f}%)")
print(f"  Precision : {final_prec:.4f}")
print(f"  Recall    : {final_rec:.4f}")
print(f"  F1-Score  : {final_f1:.4f}")
print("────────────────────────────────────────────────────────")

# ── Cell 18: Plots ────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(
    "FedOpt (FedAdam) — Tomato Leaf Disease (ResNet18)\n"
    f"Clients={NUM_CLIENTS}, α={DIRICHLET_ALPHA}, "
    f"Rounds={GLOBAL_ROUNDS}, LocalEpochs={LOCAL_EPOCHS}, "
    f"ServerLR={SERVER_LR}",
    fontsize=13, fontweight='bold')

metrics = [
    ("accuracy",  "Accuracy",  "royalblue"),
    ("precision", "Precision", "darkorange"),
    ("recall",    "Recall",    "green"),
    ("f1",        "F1-Score",  "red"),
]
for ax, (key, label, color) in zip(axes.flatten(), metrics):
    ax.plot(history["round"], history[key],
            color=color, linewidth=1.8)
    ax.set_title(label, fontsize=11)
    ax.set_xlabel("Communication Round")
    ax.set_ylabel(label)
    ax.set_xlim(1, GLOBAL_ROUNDS)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.3f'))
    ax.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.savefig("fedopt_tomatoleaf_metrics.png", dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved → fedopt_tomatoleaf_metrics.png")

# ── Cell 19: Per-client performance ──────────────────────────
print("\n── Per-Client Validation Performance (Best Model) ──────")
for c in range(NUM_CLIENTS):
    _, val_loader = client_loaders[c]
    acc, prec, rec, f1 = evaluate(global_model, val_loader)
    print(f"  Client {c+1}: Acc={acc:.4f} | Prec={prec:.4f} | "
          f"Rec={rec:.4f} | F1={f1:.4f}")

# ── Cell 20: Config summary ───────────────────────────────────
print("\n── Experiment Configuration ─────────────────────────────")
for k, v in {
    "Dataset"          : "Tomato Leaf Disease (kaustubhb999)",
    "Model"            : "ResNet18",
    "FL Algorithm"     : "FedOpt (FedAdam)",
    "Client Optimizer" : f"SGD (lr={CLIENT_LR}, momentum={CLIENT_MOMENTUM})",
    "Server Optimizer" : f"Adam (η={SERVER_LR}, β₁={BETA1}, β₂={BETA2}, τ={TAU})",
    "Num Clients"      : NUM_CLIENTS,
    "Dirichlet Alpha"  : DIRICHLET_ALPHA,
    "Local Epochs"     : LOCAL_EPOCHS,
    "Global Rounds"    : GLOBAL_ROUNDS,
    "Batch Size"       : BATCH_SIZE,
    "Num Classes"      : NUM_CLASSES,
    "Total Samples"    : len(all_samples),
    "Best Accuracy"    : f"{best_acc*100:.2f}%",
    "Final F1-Score"   : f"{final_f1:.4f}",
}.items():
    print(f"  {k:<28}: {v}")